<a href="https://colab.research.google.com/github/Brkoszov/Python-DOOM/blob/master/stable/stable_diffusion_webui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# 1. Přesun do adresáře /content
%cd /content

# 2. Instalace systémových závislostí
!apt -y update -qq
!apt -y install -qq aria2 git-lfs

# 3. Klonování ComfyUI repozitáře
comfyui_path = "/content/ComfyUI"
if os.path.exists(comfyui_path):
    print(f"Adresář {comfyui_path} již existuje. Přecházím do něj.")
    %cd {comfyui_path}
    # Volitelně můžete zde přidat !git pull, pokud chcete ComfyUI aktualizovat
else:
    print(f"Klonuji ComfyUI do {comfyui_path}...")
    !git clone https://github.com/comfyanonymous/ComfyUI
    %cd {comfyui_path}

# 4. Instalace Python závislostí
# ComfyUI často funguje dobře s novějšími PyTorch verzemi, ale je dobré se držet
# stabilní verze kompatibilní s GPU v Colabu (např. cu121)
# Zde aktualizujeme PyTorch na verzi 2.4.0, která řeší problém s 'torch.uint64'
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121 -U
!pip install xformers==0.0.28.post1 --index-url https://download.pytorch.org/whl/cu121 -U

# Instalace specifických ComfyUI závislostí
!pip install -r requirements.txt

# 5. Vytvoření adresářů pro modely, pokud neexistují
!mkdir -p models/checkpoints
!mkdir -p models/loras
!mkdir -p models/vae
!mkdir -p models/upscalers
!mkdir -p custom_nodes

# 6. Stažení základního SDXL modelu (volitelné, můžete nahrát vlastní)
# SDXL Base 1.0
# !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors -d models/checkpoints -o sd_xl_base_1.0.safetensors

# 7. Klonování rozšíření (např. ComfyUI-Manager pro snadnou správu uzlů)
manager_path = "custom_nodes/ComfyUI-Manager"
if not os.path.exists(manager_path):
    print(f"Klonuji ComfyUI-Manager do {manager_path}...")
    !git clone https://github.com/ltdrdata/ComfyUI-Manager {manager_path}
else:
    print(f"ComfyUI-Manager již existuje v {manager_path}. Klonování přeskočeno.")

In [ ]:
# Připojení Google Disku pro snadné ukládání a načítání modelů/výsledků
from google.colab import drive
drive.mount('/content/drive')

# Vytvoření symbolického odkazu na disk pro ComfyUI (volitelné, usnadní přístup)
# Můžete si vytvořit složku 'ComfyUI_models' na svém Google Disku a sem ji připojit
# !ln -s /content/drive/MyDrive/ComfyUI_models /content/ComfyUI/models/custom_models


In [ ]:
# 8. Spuštění ComfyUI
# --listen: Zpřístupní UI přes veřejnou URL (Colab automaticky generuje Gradio/Colab tunnel)
# --port 8188: Nastaví port, na kterém ComfyUI běží (volitelné)
# ComfyUI automaticky detekuje a používá xformers, pokud jsou nainstalovány, není potřeba speciální flag
%cd /content/ComfyUI
!python main.py --listen --disable-auto-launch